# Relax Structure with MatterSim — M3GNet Interatomic Potential

This notebook demonstrates structural relaxation using the **MatterSim** model,
a deep learning atomistic model based on the M3GNet architecture (Microsoft Research).

## 1. Set Input Parameters
### 1.1. Structure and Relaxation

In [ ]:
FOLDER = "uploads"
STRUCTURE_NAME = "Interface"  # Name of the structure to load from local file

RELAXATION_PARAMETERS = {
    "FMAX": 0.05,
}

In [ ]:
MATTERSIM_MODEL_PATH = "/drive/packages/models/mattersim-v1.0.0-1M.pth"

## 2. Install Packages

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples|torch|mattersim")

In [ ]:
from mat3ra.notebooks_utils.pyodide.packages.torch import apply_all_patches

apply_all_patches(include_mattersim=True)

## 3. Load Materials

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.standata.materials import Materials

structure = load_material_from_folder(FOLDER, STRUCTURE_NAME) or Material.create(
    Materials.get_by_name_first_match(STRUCTURE_NAME))

print(f"INFO: Found: '{structure.name}'")

### 3.1. Visualize Input Structure

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

visualize(structure, repetitions=[1, 1, 1], rotation="-90x")

## 4. Apply Relaxation
### 4.1. Load MatterSim Model and Create Calculator

In [ ]:
from mattersim.forcefield import MatterSimCalculator

calculator = MatterSimCalculator.from_checkpoint(
    load_path=MATTERSIM_MODEL_PATH,
    device="cpu",
)

print(f"MatterSim model loaded from {MATTERSIM_MODEL_PATH}")

### 4.2. Relax with MatterSim

In [ ]:
import plotly.graph_objs as go
from IPython.display import display
from plotly.subplots import make_subplots

from mat3ra.made.tools.convert import to_ase
from ase.optimize import BFGS

ase_structure = to_ase(structure)
ase_structure.calc = calculator
dyn = BFGS(ase_structure)

steps = []
energies = []

fig = make_subplots(rows=1, cols=1, specs=[[{"type": "scatter"}]])
scatter = go.Scatter(x=[], y=[], mode="lines+markers", name="Energy")
fig.add_trace(scatter)
fig.update_layout(title_text="Real-time Optimization Progress", xaxis_title="Step", yaxis_title="Energy (eV)")

try:
    f = go.FigureWidget(fig)
except ImportError:
    f = go.Figure(fig)
display(f)


def plotly_callback():
    step = dyn.nsteps
    energy = ase_structure.get_total_energy()
    steps.append(step)
    energies.append(energy)
    print(f"Step: {step}, Energy: {energy:.4f} eV")
    if hasattr(f, "batch_update"):
        with f.batch_update():
            f.data[0].x = steps
            f.data[0].y = energies
    else:
        f.data[0].x = steps
        f.data[0].y = energies


dyn.attach(plotly_callback, interval=1)
dyn.run(fmax=RELAXATION_PARAMETERS["FMAX"])

ase_original_structure = to_ase(structure)
ase_original_structure.calc = calculator
ase_final_structure = ase_structure

original_energy = ase_original_structure.get_total_energy()
relaxed_energy = ase_structure.get_total_energy()

print(f"The final energy is {float(relaxed_energy):.3f} eV.")

## 5. Analyze Results
### 5.1. View Structure Before and After Relaxation

In [ ]:
from mat3ra.made.tools.convert import from_ase

material_original = Material.create(from_ase(ase_original_structure))
material_relaxed = Material.create(from_ase(ase_final_structure))
material_original.name = structure.name
material_relaxed.name = structure.name + " (MatterSim Relaxed)"

visualize([material_original, material_relaxed], viewer=ViewersEnum.wave)

### 5.2. Output interlayer distance before and after relaxation

In [ ]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance

SUBSTRATE_TAG = 0
FILM_TAG = 1

print(
    f"Interlayer distance before relaxation: {get_average_interlayer_distance(material_original, SUBSTRATE_TAG, FILM_TAG):.4f} Å")
print(
    f"Interlayer distance after relaxation:  {get_average_interlayer_distance(material_relaxed, SUBSTRATE_TAG, FILM_TAG):.4f} Å")

## References

[1] MatterSim: https://github.com/microsoft/mattersim

[2] Han Yang et al., "MatterSim: A Deep Learning Atomistic Model Across Elements, Temperatures and Pressures," arXiv:2405.04967 (2024)